# OneVoice V2 — Đánh giá SenseVoice English đã fine-tune

Notebook này không train và không thay runtime ONNX. Nó benchmark checkpoint cuối model.pt.ep10 trên test clean/noisy. Cần GPU. Không dùng model.pt.avg3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
MANIFEST = MYDRIVE / 'onevoice_audio_v2_1/manifest.jsonl'
CHECKPOINT = WORK_ROOT / 'models/sensevoice_en_construction_v1/model.pt.ep10'
REPORT_ROOT = WORK_ROOT / 'reports/en_asr_finetuned_v1'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MODELSCOPE_CACHE'] = str(WORK_ROOT / 'model_cache/modelscope')
# Install FunASR first: its resolver may otherwise upgrade torch after we pin it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'funasr>=1.4.3', 'modelscope', 'soundfile'], check=True)
# Pin the final, ABI-matched CUDA 13.0 pair.  TorchAudio 2.13 does not exist, so it cannot be used with Colab's transient torch 2.13.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'torch==2.10.0', 'torchaudio==2.10.0', '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
import torch, torchaudio
if not (torch.__version__.startswith('2.10.0') and torchaudio.__version__.startswith('2.10.0')):
    raise RuntimeError(f'PyTorch ABI pin failed: torch={torch.__version__}, torchaudio={torchaudio.__version__}. Restart runtime, then rerun this setup cell once.')
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trước khi đánh giá checkpoint SenseVoice.')
if not MANIFEST.is_file() or not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Missing manifest or final checkpoint: {MANIFEST}, {CHECKPOINT}')
print('GPU:', torch.cuda.get_device_name(0))
print('Checkpoint:', CHECKPOINT, f'({CHECKPOINT.stat().st_size / 1024**2:.1f} MB)')

def run_streaming(command, label):
    print(f'[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'{label} failed with exit code {code}; the complete traceback is printed above.')


In [ ]:
# Smoke test: kiểm tra checkpoint có nạp và decode được trước khi chạy toàn bộ test.
smoke_dir = REPORT_ROOT / 'smoke_clean'
if not (smoke_dir / 'aggregate.json').is_file():
    run_streaming([sys.executable, 'scripts/benchmark_sensevoice_checkpoint.py', str(MANIFEST), '--checkpoint', str(CHECKPOINT), '--audio', 'clean', '--split', 'test', '--max-samples', '16', '--progress-every', '4', '--report-dir', str(smoke_dir)], 'SenseVoice smoke clean')
else:
    print('Smoke test already completed; skipping.')


In [ ]:
# Full held-out test. Existing complete reports are retained; rerun safely after a runtime/account change.
for audio in ('clean', 'noisy'):
    report_dir = REPORT_ROOT / audio
    if all((report_dir / name).is_file() for name in ('aggregate.json', 'predictions.csv', 'run_manifest.json')):
        print(f'{audio} report already complete; skipping.')
        continue
    run_streaming([sys.executable, 'scripts/benchmark_sensevoice_checkpoint.py', str(MANIFEST), '--checkpoint', str(CHECKPOINT), '--audio', audio, '--split', 'test', '--progress-every', '25', '--report-dir', str(report_dir)], f'SenseVoice full {audio}')


In [ ]:
results = {audio: json.loads((REPORT_ROOT / audio / 'aggregate.json').read_text(encoding='utf-8')) for audio in ('clean', 'noisy')}
display(results)
print('Chỉ export/replace ONNX nếu WER/CER, critical-term recall và clean/noisy regression tốt hơn baseline.')
